# Atividade Ponderada: MLP do Zero em NumPy

Este notebook documenta a construcao e a melhoria de um Multi-Layer Perceptron para classificar digitos do MNIST sem usar frameworks de deep learning para a matematica da rede. O objetivo nao e esconder a complexidade atras de uma API pronta: e abrir o mecanismo em partes pequenas, validar cada parte e so depois melhorar o modelo.

Resultado de maior acuracia registrado em `results/summary.json`: **98.16% de acuracia no teste**, mas com overfitting leve. A resolucao recomendada usa a mesma arquitetura `784 -> 256 -> 128 -> 10`, com SGD momentum, `L2=1e-3` e early stopping por `val_data_loss`, chegando a **97.86%** com menor gap treino-validacao.


## Fontes de estudo usadas

Usei duas referencias iniciais para organizar o raciocinio:

- [Teaching a Perceptron by Hand](https://thomascountz.com/2018/03/26/perceptrons-implementing-and-part-1): ajudou a partir do caso mais simples, onde pesos e vies fazem uma decisao binaria.
- [Parameter optimization in neural networks](https://www.deeplearning.ai/ai-notes/optimization/index.html): fundamentou a separacao entre modelo, loss/custo, gradiente, learning rate e batch size.

A melhoria do modelo segue a mesma logica: alterar capacidade, otimizacao e regularizacao, medir o efeito e comparar contra a configuracao anterior. Quando apareceu overfitting, a resolucao tambem virou experimento: early stopping e L2 mais forte, com o trade-off documentado.


## Processo de desenvolvimento

1. Ler o enunciado e transformar requisitos em estrutura de repositorio.
2. Implementar o forward pass para qualquer numero de camadas.
3. Implementar softmax + cross-entropy de forma estavel.
4. Implementar backpropagation manual.
5. Validar gradientes numericamente em uma rede pequena.
6. Treinar duas configuracoes iniciais no MNIST.
7. Melhorar o modelo com mais capacidade, momentum e L2 leve.
8. Investigar o overfitting gerado pela melhoria.
9. Resolver com L2 mais forte e early stopping por `val_data_loss`.
10. Comparar resultados e registrar decisoes, dificuldades e evidencias.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mlp import MLPClassifier, gradient_check, summarize_gradient_check
from mlp.data import load_mnist
from mlp.losses import softmax

RESULTS = ROOT / 'results'
print('Repo:', ROOT)
print('Resultados disponiveis:', sorted(path.name for path in RESULTS.glob('*')))


## 1. Dados: MNIST como matriz NumPy

O enunciado permite carregar o MNIST via Keras ou Torchvision. Neste projeto, `torchvision.datasets.MNIST` faz apenas o download e a leitura dos arquivos. A partir dai, as imagens viram arrays NumPy `float32` normalizados entre 0 e 1.

Cada imagem `28 x 28` e achatada para um vetor de `784` atributos. A rede final recebe entao uma matriz `X` com forma `(n_exemplos, 784)` e rotulos `y` com valores de 0 a 9.


In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist(ROOT / 'data', train_limit=1000, test_limit=200)
print('Treino:', X_train.shape, y_train.shape)
print('Validacao:', X_val.shape, y_val.shape)
print('Teste:', X_test.shape, y_test.shape)
print('Min/max dos pixels normalizados:', X_train.min(), X_train.max())


## 2. Arquitetura final e resolucao

A primeira versao forte usava:

```text
784 -> 128 ReLU -> 64 ReLU -> 10
```

A melhoria focada no modelo passou para:

```text
784 -> 256 ReLU -> 128 ReLU -> 10
```

A mudanca aumentou a capacidade da rede, manteve duas camadas ocultas e adicionou **SGD com momentum 0.9**. O problema encontrado foi que `L2=1e-4` ainda permitiu treino praticamente perfeito. A resolucao recomendada manteve a arquitetura, mas usou **L2 mais forte (`1e-3`)** e **early stopping por `val_data_loss`**, restaurando o melhor checkpoint.


In [ ]:
model = MLPClassifier([784, 256, 128, 10], activation='relu', seed=84)
logits = model.forward(X_train[:8])
probs = softmax(logits)
print('Logits:', logits.shape)
print('Probabilidades:', probs.shape)
print('Soma por exemplo:', probs.sum(axis=1))


## 3. Forward pass

Cada camada escondida calcula:

```text
Z[l] = A[l-1] @ W[l] + b[l]
A[l] = ReLU(Z[l])
```

A ultima camada nao aplica ReLU. Ela produz logits, que depois passam por softmax. Essa separacao evita misturar a camada linear final com a interpretacao probabilistica.


In [ ]:
import inspect
from mlp.network import MLPClassifier

print(inspect.getsource(MLPClassifier.forward))


## 4. Backpropagation e gradient check reutilizavel

O ponto mais sensivel foi garantir que os gradientes tinham dimensoes e sinais corretos. Para softmax com cross-entropy, o gradiente dos logits fica:

```text
dZ_final = (probabilidades - y_one_hot) / batch_size
```

Agora o gradient check virou uma funcao reutilizavel em `mlp/gradient_check.py`, nao apenas codigo solto no teste. Isso deixa a validacao mais facil de repetir quando a arquitetura muda.


In [ ]:
import numpy as np

rng = np.random.default_rng(13)
X_small = rng.normal(size=(5, 4)).astype(np.float32)
y_small = np.array([0, 1, 2, 1, 0])
small_model = MLPClassifier([4, 6, 5, 3], activation='tanh', seed=3)
checks = gradient_check(small_model, X_small, y_small, max_checks_per_parameter=2, seed=9)
summary_grad = summarize_gradient_check(checks)
print(json.dumps({k: v for k, v in summary_grad.items() if k != 'checks'}, indent=2))
assert summary_grad['passed']


## 5. Treinamento por mini-batches com SGD

O treinamento usa mini-batches de 128 exemplos. A cada batch:

1. forward pass calcula logits;
2. softmax + cross-entropy calcula loss;
3. backprop calcula gradientes;
4. SGD atualiza `W` e `b` na direcao oposta ao gradiente.

Na melhoria do modelo, o SGD usa momentum. A ideia e acumular parte da direcao anterior para reduzir zigue-zague e acelerar convergencia. Como o modelo melhorado usa L2, o relatorio separa cross-entropy pura (`data_loss`) da penalidade de regularizacao. Isso evita interpretar uma loss total maior como pior generalizacao quando parte dela e apenas penalidade sobre pesos.

Para reproduzir o treino completo, rode no terminal a partir da raiz do repo:

```powershell
python scripts/train.py
```

O notebook le os resultados ja versionados para evitar retreinar tudo a cada abertura.


In [ ]:
summary_path = RESULTS / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
for row in summary:
    print(
        f"{row['name']}: test_acc={row['test_accuracy']:.4f}, "
        f"test_ce={row['test_data_loss']:.4f}, obj_loss={row['test_loss']:.4f}, "
        f"momentum={row['momentum']}, layers={row['layer_sizes']}"
    )

best = max(summary, key=lambda row: row['test_accuracy'])
resolved = next(row for row in summary if row['name'].startswith('resolved_'))
assert best['test_accuracy'] >= 0.98
assert resolved['test_accuracy'] >= 0.975
print('\nMaior acuracia:', best['name'], f"{best['test_accuracy']:.2%}")
print('Modelo recomendado:', resolved['name'], f"{resolved['test_accuracy']:.2%}")


## 6. Comparacao de configuracoes

| Configuracao | Arquitetura | Momentum | L2 | Epocas | Acuracia teste | CE teste |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `compact_relu_64_32` | 784 -> 64 -> 32 -> 10 | 0.0 | 0 | 8 | 96.34% | 0.1246 |
| `final_relu_128_64` | 784 -> 128 -> 64 -> 10 | 0.0 | 0 | 10 | 97.59% | 0.0821 |
| `improved_relu_256_128_momentum` | 784 -> 256 -> 128 -> 10 | 0.9 | 1e-4 | 15 | **98.16%** | **0.0607** |
| `resolved_relu_256_128_l2_earlystop` | 784 -> 256 -> 128 -> 10 | 0.9 | 1e-3 | 11/25 | **97.86%** | 0.0694 |

A melhoria de maior acuracia foi de **+0.57 ponto percentual** em teste sobre a configuracao anterior. A resolucao do overfitting troca parte desse ganho por estabilidade: fica **+0.27 ponto percentual** acima do modelo anterior e reduz o gap treino-validacao.


## 7. Curvas de loss/acuracia

![Curvas de treino e validacao](../results/loss_accuracy.png)


## 8. Matriz de confusao do melhor modelo

![Matriz de confusao do modelo final](../results/confusion_matrix_final.png)

Os erros restantes aparecem em pares visualmente parecidos. Um MLP simples olha para pixels achatados, entao nao aproveita estrutura espacial como uma CNN aproveitaria.


## 9. Investigacao de overfitting

A suspeita veio do comportamento do modelo melhorado: a acuracia de treino chegou praticamente a 100%, enquanto a validacao ficou perto de 98%. Para investigar e resolver, medi tres sinais:

1. gap final entre treino e validacao;
2. queda da validacao final em relacao ao melhor epoch;
3. aumento da cross-entropy de validacao desde o minimo.

O relatorio completo esta em `results/overfitting_analysis.md`.


In [ ]:
overfit = json.loads((RESULTS / 'overfitting_analysis.json').read_text(encoding='utf-8'))
print(overfit['diagnosis']['conclusion'])
for item in overfit['models']:
    print(
        f"{item['name']}: gap={item['final_generalization_gap_pp']:.2f} p.p., "
        f"queda_val={item['final_val_accuracy_drop_from_best_pp']:.2f} p.p., "
        f"selected_epoch={item.get('selected_epoch')}, diag={item['diagnosis']}"
    )


![Sinais de overfitting](../results/overfitting_gaps.png)

Conclusao: o problema encontrado foi **overfitting leve** no modelo `improved`. A resolucao foi implementar early stopping com restauracao do melhor checkpoint e aumentar L2 para `1e-3`. Com isso, o gap caiu de 2.03 p.p. para 1.20 p.p.; a acuracia final recomendada ficou em 97.86%, ainda acima do modelo anterior de 97.59%.


## 10. Decisoes e dificuldades

A decisao tecnica mais importante foi nao ir direto para o MNIST sem validacao. Primeiro validei forward, depois gradient check, depois um problema pequeno, e so entao rodei MNIST. Isso evitou confundir erro de gradiente com problema de hiperparametro.

Na melhoria, foquei no modelo: aumentei largura, ativei momentum e adicionei L2 leve. O ganho de acuracia mostrou que havia margem, mas a diferenca entre treino perfeito e validacao/teste revelou o problema encontrado: overfitting leve. A resolucao escolhida foi `L2=1e-3` + early stopping por `val_data_loss`. Eu aceitei perder 0.30 p.p. em relacao ao pico de teste para reduzir memorizacao e manter resultado acima do modelo anterior.

Se eu fosse continuar melhorando, testaria dropout manual e data augmentation simples. Para a ponderada, mantive o foco no MLP exigido e registrei o trade-off de generalizacao.


## 11. Conclusao

A entrega cumpre os requisitos obrigatorios e ainda melhora o modelo inicial: MLP com duas camadas ocultas, ReLU configuravel, softmax + cross-entropy, backpropagation manual, mini-batch SGD, comparacao de configuracoes, curvas de treino e acuracia final maior que 98% no teste.
